# 03 — Multi-parameter team model + TI simulation

1. **Elo core** on the game tape (idle-decay as roster-churn proxy) with a
   walk-forward Brier/log-loss check against the 50% baseline.
2. **Feature layer**: lane-phase strength (gold@10), comeback/throw rates,
   draft diversity, side bias — a logistic stack when sklearn is present.
3. **Bracket Monte Carlo** of a TI-like group→double-elim format with Elo
   noise (`elo_sigma≈60`) to model the two-month gap to the main event.
Output: `data/model_pchamp.parquet` — P(champion) per team with CIs.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
from src.model import fit_elo, evaluate, simulate_ti
from src.features import team_features, draft_features
matches = pd.read_parquet(DATA / 'matches.parquet')
elo, hist = fit_elo(matches)
print('holdout eval:', evaluate(hist))
teams_meta = pd.read_parquet(DATA / 'teams.parquet')[['team_id','name']]
elo_named = elo.rename_axis('team_id').reset_index().merge(teams_meta, on='team_id')
elo_named.nlargest(20, 'elo')

In [ ]:
# Feature layer + optional logistic stacking over Elo
tf = team_features(pd.read_parquet(DATA / 'match_players.parquet'), matches)
df_feats = draft_features(pd.read_parquet(DATA / 'picks_bans.parquet'), matches)
team_table = (elo_named.merge(tf, on='team_id', how='left')
                       .merge(df_feats, on='team_id', how='left'))
try:
    from sklearn.linear_model import LogisticRegression
    m = matches.dropna(subset=['radiant_team_id','dire_team_id']).sort_values('start_time')
    j = (m.merge(team_table.add_prefix('r_'), left_on='radiant_team_id', right_on='r_team_id')
          .merge(team_table.add_prefix('d_'), left_on='dire_team_id', right_on='d_team_id'))
    X = pd.DataFrame({
        'elo_gap': j.r_elo - j.d_elo,
        'gold10_gap': (j.r_gold10_mean - j.d_gold10_mean).fillna(0),
        'comeback_gap': (j.r_comeback_rate - j.d_comeback_rate).fillna(0),
        'draft_div_gap': (j.d_draft_hhi - j.r_draft_hhi).fillna(0),
    })
    y = j.radiant_win.astype(int)
    cut = int(len(X)*0.75)
    lr = LogisticRegression().fit(X[:cut], y[:cut])
    p = lr.predict_proba(X[cut:])[:,1]
    print('stacked holdout Brier:', float(((p - y[cut:])**2).mean()))
    print(dict(zip(X.columns, lr.coef_[0].round(4))))
except ImportError:
    print('sklearn not installed - using pure Elo')

In [ ]:
# TI simulation: EDIT this list to the final TI-2026 participant set.
TI_TEAMS = None  # e.g. ['Team Spirit','Team Liquid','Gaimin Gladiators', ...]
pool = elo_named.nlargest(16, 'elo') if TI_TEAMS is None else \
       elo_named[elo_named.name.isin(TI_TEAMS)]
sim = simulate_ti(list(pool.name), dict(zip(pool.name, pool.elo)), n_sims=20000)
sim.to_parquet(DATA / 'model_pchamp.parquet')
sim

**Reading the output:** `elo_sigma=60` injects ~2 months of patch/roster
uncertainty; the random group draw widens CIs further. Treat `p_champion`
as a *band* (see ci95 columns), not a point estimate — and re-run after
every tier-1 event and after the pre-TI patch drops.